# RSNA Knee MARS 3D foundation-model validation
Private attributed volumetric MRI representation comparison. The 58 gold studies remain evaluation-only, and this notebook cannot submit.


In [ ]:
"""Leakage-safe paired test of physically canonical knee-MRI preprocessing.

Public assets:
* RadImageNet ResNet-50 checkpoint, SHA-256 pinned below.
* Steven Lee Hans' v4 report-derived labels.

All 58 organizer-labelled studies are excluded from training, validation,
early stopping, and model selection. This script cannot create a submission.
"""

from __future__ import annotations

import gc
import hashlib
import json
import math
import os
import random
import re
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, TensorDataset
from torchvision.models import resnet50


TARGETS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
           "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
           "Contusion", "Fracture"]
PLANES = ("Sagittal", "Coronal", "Axial")
SLOTS = (("Sagittal", True), ("Coronal", True), ("Axial", True),
         ("Sagittal", False), ("Coronal", False), ("Axial", False))
N_SLICE = 3
IMG = 224
CROP_MM = 130.0
TOKEN_DIM = 2048
SEEDS = (20260901, 20260902, 20260903, 20260904, 20260905)
RAD_SHA = "08629f7e7bd3e29b8ee9522ca3f65ce4d010a7ddf74f0ea3c7e3f3d0bbab0734"
FATSAT = re.compile(r"\bfs\b|fat\s*sat|fsat|stir|spair|spir|water\s*excit|fatsup", re.I)
PD = re.compile(r"\bpd\b|\bpdw\b|proton|density|\bdp\b", re.I)
T1 = re.compile(r"\bt1\b|\bt1w\b", re.I)
T2 = re.compile(r"\bt2\b|\bt2w\b", re.I)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
START = time.time()


def log(message):
    print(f"[{time.time() - START:7.1f}s] {message}", flush=True)


def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)


def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(8 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def competition_root():
    for path in (Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
                 Path("/kaggle/input/rsna-knee-abnormality-detection")):
        if (path / "train.csv").is_file(): return path
    raise FileNotFoundError("competition data is not attached")


def unique_input(name):
    matches = list(Path("/kaggle/input").rglob(name))
    if len(matches) != 1:
        raise FileNotFoundError(f"expected one {name}, found {matches}")
    return matches[0]


def ffloat(value):
    try: value = float(value)
    except Exception: return None
    return value if math.isfinite(value) else None


def vec(value, n):
    try: value = np.asarray([float(x) for x in value], np.float64)
    except Exception: return None
    return value if len(value) >= n and np.isfinite(value[:n]).all() else None


def classify(ds):
    text = " ".join(str(getattr(ds, key, "") or "") for key in
                    ("SeriesDescription", "SequenceName", "ScanOptions", "ScanningSequence"))
    tr, te = ffloat(getattr(ds, "RepetitionTime", None)), ffloat(getattr(ds, "EchoTime", None))
    fs = bool(FATSAT.search(text))
    if T1.search(text) and not T2.search(text) and not PD.search(text): weight = "T1"
    elif T2.search(text) and not PD.search(text): weight = "T2"
    elif PD.search(text): weight = "PD"
    elif tr is not None and tr < 800: weight = "T1"
    elif te is not None and te > 60: weight = "T2"
    elif tr is not None and tr >= 800: weight = "PD"
    else: weight = "UNK"
    return weight, fs


def inspect_series(item):
    study, uid, plane, folder = item
    files = sorted(Path(folder).glob("*.dcm"))
    if not files: return None
    try:
        ds = pydicom.dcmread(files[len(files)//2], stop_before_pixels=True, force=True)
    except Exception: return None
    weight, fs = classify(ds)
    ps = vec(getattr(ds, "PixelSpacing", None), 2)
    iop = vec(getattr(ds, "ImageOrientationPatient", None), 6)
    ipp = vec(getattr(ds, "ImagePositionPatient", None), 3)
    rows, cols = ffloat(getattr(ds, "Rows", None)), ffloat(getattr(ds, "Columns", None))
    centre_x = None
    if ps is not None and iop is not None and ipp is not None and rows and cols:
        centre_x = float((ipp + iop[:3]*ps[1]*cols/2 + iop[3:]*ps[0]*rows/2)[0])
    tag = str(getattr(ds, "ImageLaterality", "") or getattr(ds, "Laterality", "")).upper()
    tag_lat = tag[:1] if tag[:1] in ("L", "R") else None
    geometry_lat = None if centre_x is None or abs(centre_x) < 20 else ("R" if centre_x < 0 else "L")
    # Geometry is canonical when decisive; the DICOM tag is only a fallback.
    lat = geometry_lat or tag_lat
    return {"study": study, "uid": uid, "plane": plane, "folder": str(folder),
            "files": [str(x) for x in files], "n": len(files), "weight": weight,
            "fs": fs, "ps": ps.tolist() if ps is not None else None, "lat": lat}


def choose(records, plane, fluid, canonical):
    candidates = [r for r in records if r["plane"] == plane]
    if canonical:
        if fluid:
            ranked = [(0 if r["weight"] == "PD" and r["fs"] else
                       1 if r["weight"] == "T2" and r["fs"] else
                       2 if r["fs"] else 9, -r["n"], r) for r in candidates]
        else:
            ranked = [(0 if r["weight"] == "PD" and not r["fs"] else
                       1 if r["weight"] == "T1" and not r["fs"] else
                       2 if r["weight"] == "T2" and not r["fs"] else 9,
                       -r["n"], r) for r in candidates]
        ranked = [x for x in ranked if x[0] < 9]
        return sorted(ranked, key=lambda x: (x[0], x[1]))[0][2] if ranked else None
    preferred = [r for r in candidates if bool(r["fs"]) == bool(fluid)]
    pool = preferred or candidates
    return max(pool, key=lambda r: r["n"]) if pool else None


def ordered_headers(record):
    rows = []
    for i, path in enumerate(record["files"]):
        try:
            ds = pydicom.dcmread(path, stop_before_pixels=True, force=True,
                                 specific_tags=["ImageOrientationPatient", "ImagePositionPatient", "InstanceNumber"])
            iop, ipp = vec(getattr(ds, "ImageOrientationPatient", None), 6), vec(getattr(ds, "ImagePositionPatient", None), 3)
            projection = float(np.dot(ipp, np.cross(iop[:3], iop[3:]))) if iop is not None and ipp is not None else None
            instance = ffloat(getattr(ds, "InstanceNumber", None))
        except Exception: projection, instance = None, None
        rows.append((path, projection, instance, i))
    if all(r[1] is not None for r in rows): rows.sort(key=lambda r: r[1])
    elif all(r[2] is not None for r in rows): rows.sort(key=lambda r: r[2])
    return rows


def selected_rows(record, canonical):
    rows = ordered_headers(record)
    if not rows: return []
    if not canonical:
        indices = np.linspace(.20*(len(rows)-1), .80*(len(rows)-1), N_SLICE).round().astype(int)
        return [rows[i] for i in indices]
    positions = np.asarray([r[1] if r[1] is not None else np.nan for r in rows])
    if np.isfinite(positions).all() and len(rows) > 1:
        centre = float(np.median(positions)); span = 35.0 if record["plane"] != "Axial" else 24.0
        targets = np.linspace(centre-span, centre+span, N_SLICE)
        return [rows[int(np.argmin(abs(positions-target)))] for target in targets]
    indices = np.linspace(.10*(len(rows)-1), .90*(len(rows)-1), N_SLICE).round().astype(int)
    return [rows[i] for i in indices]


def read_pixels(path):
    ds = pydicom.dcmread(path, force=True)
    image = ds.pixel_array.astype(np.float32)
    image = image * float(getattr(ds, "RescaleSlope", 1) or 1) + float(getattr(ds, "RescaleIntercept", 0) or 0)
    if str(getattr(ds, "PhotometricInterpretation", "")) == "MONOCHROME1": image = image.max() - image
    ps = vec(getattr(ds, "PixelSpacing", None), 2)
    return image, ps


def render_slot(record, canonical):
    output = np.zeros((N_SLICE, IMG, IMG), np.uint8)
    if record is None: return output, 0
    arrays = []
    for row in selected_rows(record, canonical):
        try: arrays.append(read_pixels(row[0]))
        except Exception: arrays.append((None, None))
    valid = [a for a, _ in arrays if a is not None]
    if not valid: return output, 0
    values = np.concatenate([a.ravel()[::max(1, a.size//20000)] for a in valid])
    if canonical:
        centre = float(np.median(values)); scale = 1.4826*float(np.median(abs(values-centre)))
        if scale < 1e-6: scale = max(float(np.std(values)), 1e-6)
        low, high = centre-4*scale, centre+4*scale
    else:
        low, high = np.percentile(values, [1, 99])
    for i, (image, ps) in enumerate(arrays):
        if image is None: continue
        h, w = image.shape
        if ps is None: ps = np.asarray((.5, .5))
        if canonical:
            crop_h = min(h, max(16, int(round(CROP_MM/max(ps[0], 1e-3)))))
            crop_w = min(w, max(16, int(round(CROP_MM/max(ps[1], 1e-3)))))
        else:
            crop_h = crop_w = min(h, w, max(16, int(round(CROP_MM/max(ps[0], 1e-3)))))
        y0, x0 = (h-crop_h)//2, (w-crop_w)//2
        image = image[y0:y0+crop_h, x0:x0+crop_w]
        image = np.clip((image-low)/max(high-low, 1e-6), 0, 1)
        image = cv2.resize(image, (IMG, IMG), interpolation=cv2.INTER_AREA)
        if canonical and record.get("lat") == "R" and record["plane"] in ("Coronal", "Axial"):
            image = image[:, ::-1]
        output[i] = np.round(image*255).astype(np.uint8)
    return output, 1


class Encoder(nn.Module):
    def __init__(self):
        super().__init__(); self.backbone = nn.Sequential(*list(resnet50(weights=None).children())[:-2])
    def forward(self, x): return self.backbone(x).mean((2, 3))


def load_encoder():
    path = unique_input("ResNet50.pt")
    if sha256(path) != RAD_SHA: raise RuntimeError("RadImageNet checkpoint hash drift")
    model = Encoder(); state = torch.load(path, map_location="cpu", weights_only=True)
    model.load_state_dict(state, strict=True); model.eval().to(DEVICE)
    for p in model.parameters(): p.requires_grad_(False)
    return model


def encode_arm(studies, by_study, canonical):
    features = np.zeros((len(studies), len(SLOTS)*N_SLICE, TOKEN_DIM), np.float16)
    masks = np.zeros((len(studies), len(SLOTS)*N_SLICE), np.float32)
    encoder = load_encoder(); batch_study = 24
    for start in range(0, len(studies), batch_study):
        ids = studies[start:start+batch_study]
        jobs = []
        for uid in ids:
            records = by_study.get(uid, [])
            jobs.extend((choose(records, plane, fluid, canonical), canonical) for plane, fluid in SLOTS)
        with ThreadPoolExecutor(max_workers=12) as pool: rendered = list(pool.map(lambda x: render_slot(*x), jobs))
        pixels = np.stack([x[0] for x in rendered]).reshape(len(ids), len(SLOTS)*N_SLICE, IMG, IMG)
        slot_ok = np.asarray([x[1] for x in rendered], np.float32).reshape(len(ids), len(SLOTS))
        token_ok = np.repeat(slot_ok, N_SLICE, axis=1)
        flat = pixels.reshape(-1, IMG, IMG); valid = np.flatnonzero(token_ok.reshape(-1)>0)
        encoded = np.zeros((len(flat), TOKEN_DIM), np.float16)
        for b0 in range(0, len(valid), 128):
            ix = valid[b0:b0+128]
            x = torch.from_numpy(flat[ix]).to(DEVICE).float().div_(127.5).sub_(1).unsqueeze(1).expand(-1,3,-1,-1)
            with torch.autocast("cuda", enabled=DEVICE.type=="cuda"):
                y = encoder(x).float().cpu().numpy().astype(np.float16)
            encoded[ix] = y
        features[start:start+len(ids)] = encoded.reshape(len(ids), -1, TOKEN_DIM)
        masks[start:start+len(ids)] = token_ok
        if start % (batch_study*10) == 0: log(f"{'canonical' if canonical else 'incumbent'} encoded {start}/{len(studies)}")
    del encoder; gc.collect(); torch.cuda.empty_cache()
    return features, masks


class AttentionHead(nn.Module):
    def __init__(self):
        super().__init__(); self.norm=nn.LayerNorm(TOKEN_DIM); self.proj=nn.Linear(TOKEN_DIM,256)
        self.att=nn.Linear(256,len(TARGETS)); self.cls=nn.Parameter(torch.empty(len(TARGETS),256)); self.bias=nn.Parameter(torch.zeros(len(TARGETS)))
        nn.init.trunc_normal_(self.cls,std=.02)
    def forward(self, x, mask):
        h=torch.nn.functional.gelu(self.proj(self.norm(x.float()))); a=self.att(h)
        a=a.masked_fill(mask[:,:,None]<.5,-1e4); a=torch.softmax(a,dim=1)
        pooled=torch.einsum("btn,btf->bnf",a,h)
        return (pooled*self.cls).sum(-1)+self.bias


def macro_auc(y, p):
    scores=[]
    for j in range(y.shape[1]):
        truth = y[:, j]
        if not np.isin(np.unique(truth), (0.0, 1.0)).all():
            truth = (truth >= 0.5).astype(np.uint8)
        if len(np.unique(truth)) > 1:
            scores.append(roc_auc_score(truth, p[:, j]))
    return float(np.mean(scores))


def train_head(features, masks, labels, train_idx, val_idx, seed):
    seed_all(seed); model=AttentionHead().to(DEVICE); opt=torch.optim.AdamW(model.parameters(),lr=2e-4,weight_decay=1e-3)
    tx=torch.from_numpy(features[train_idx]); tm=torch.from_numpy(masks[train_idx]); ty=torch.from_numpy(labels[train_idx])
    loader=DataLoader(TensorDataset(tx,tm,ty),batch_size=64,shuffle=True,generator=torch.Generator().manual_seed(seed))
    best, best_score = None, -1
    for epoch in range(20):
        model.train()
        for x,m,y in loader:
            x,m,y=x.to(DEVICE),m.to(DEVICE),y.to(DEVICE); opt.zero_grad(set_to_none=True)
            loss=torch.nn.functional.binary_cross_entropy_with_logits(model(x,m),y.float()); loss.backward(); opt.step()
        pred=predict(model,features,masks,val_idx); score=macro_auc(labels[val_idx],pred)
        if score>best_score: best_score=score; best={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
    model.load_state_dict(best); return predict(model,features,masks,np.arange(len(features))), best_score


@torch.no_grad()
def predict(model, features, masks, indices):
    model.eval(); out=[]
    for b0 in range(0,len(indices),128):
        ix=indices[b0:b0+128]; x=torch.from_numpy(features[ix]).to(DEVICE); m=torch.from_numpy(masks[ix]).to(DEVICE)
        out.append(torch.sigmoid(model(x,m)).cpu().numpy())
    return np.concatenate(out)


def paired_bootstrap(y, inc, can, draws=5000, seed=20260901):
    rng=np.random.default_rng(seed); delta=[]
    for _ in range(draws):
        ix=rng.integers(0,len(y),len(y)); delta.append(macro_auc(y[ix],can[ix])-macro_auc(y[ix],inc[ix]))
    d=np.asarray(delta); return {"mean":float(d.mean()),"p_improve":float((d>0).mean()),"q025":float(np.quantile(d,.025)),"q975":float(np.quantile(d,.975))}


def main():
    if DEVICE.type!="cuda": raise RuntimeError("paired validation requires GPU")
    root=competition_root(); train=pd.read_csv(root/"train.csv",dtype={"StudyInstanceUID":str})
    gold=train[TARGETS].notna().all(axis=1).to_numpy();
    if int(gold.sum())!=58: raise RuntimeError(f"expected 58 gold studies, found {gold.sum()}")
    label_path=unique_input("llm_labels_v4_blend.csv"); weak=pd.read_csv(label_path,dtype={"StudyInstanceUID":str})
    aligned=train[["StudyInstanceUID"]].merge(weak,on="StudyInstanceUID",how="left",validate="one_to_one")
    labels=aligned[TARGETS].to_numpy(np.float32)
    if not np.isfinite(labels).all(): raise RuntimeError("weak labels incomplete")
    series=pd.read_csv(root/"train_series.csv",dtype=str); series_root=root/("train_series" if (root/"train_series").is_dir() else "train_images")
    items=[(r.StudyInstanceUID,r.SeriesInstanceUID,r.Anatomical_Plane,series_root/r.StudyInstanceUID/r.SeriesInstanceUID) for r in series.itertuples(index=False)]
    with ThreadPoolExecutor(max_workers=16) as pool: records=[r for r in pool.map(inspect_series,items) if r]
    by_study={}
    for record in records: by_study.setdefault(record["study"],[]).append(record)
    studies=train.StudyInstanceUID.tolist(); inc_f,inc_m=encode_arm(studies,by_study,False); can_f,can_m=encode_arm(studies,by_study,True)
    nongold=np.flatnonzero(~gold); gold_idx=np.flatnonzero(gold); results=[]; inc_gold=[]; can_gold=[]
    for seed in SEEDS:
        rng=np.random.default_rng(seed); perm=rng.permutation(nongold); cut=int(.9*len(perm)); tr,va=perm[:cut],perm[cut:]
        inc_pred,inc_weak=train_head(inc_f,inc_m,labels,tr,va,seed); can_pred,can_weak=train_head(can_f,can_m,labels,tr,va,seed)
        ia=macro_auc(train.loc[gold,TARGETS].to_numpy(float),inc_pred[gold_idx]); ca=macro_auc(train.loc[gold,TARGETS].to_numpy(float),can_pred[gold_idx])
        results.append({"seed":seed,"incumbent_weak_val_auc":inc_weak,"canonical_weak_val_auc":can_weak,"incumbent_gold_auc":ia,"canonical_gold_auc":ca,"delta":ca-ia})
        inc_gold.append(inc_pred[gold_idx]); can_gold.append(can_pred[gold_idx]); log(f"seed {seed}: gold delta {ca-ia:+.6f}")
    y_gold=train.loc[gold,TARGETS].to_numpy(float); inc_mean=np.mean(inc_gold,axis=0); can_mean=np.mean(can_gold,axis=0)
    boot=paired_bootstrap(y_gold,inc_mean,can_mean); deltas=np.asarray([x["delta"] for x in results])
    gate={"delta_at_least_0_005":float(macro_auc(y_gold,can_mean)-macro_auc(y_gold,inc_mean))>=.005,
          "positive_seeds_at_least_4":int((deltas>0).sum())>=4,"bootstrap_p_at_least_0_95":boot["p_improve"]>=.95,"bootstrap_q025_positive":boot["q025"]>0}
    report={"contract":"private paired validation; gold evaluation only; no submission","encoder":"RadImageNet ResNet-50","encoder_sha256":RAD_SHA,
            "weak_labels":"stevenleehans llm_labels_v4_blend.csv","gold_fit_count":0,"gold_evaluation_count":58,"seeds":results,
            "ensemble":{"incumbent_gold_auc":macro_auc(y_gold,inc_mean),"canonical_gold_auc":macro_auc(y_gold,can_mean),"delta":macro_auc(y_gold,can_mean)-macro_auc(y_gold,inc_mean)},
            "bootstrap":boot,"gate":gate,"passed":bool(all(gate.values())),"submission_created":False}
    Path("/kaggle/working/physical_paired_validation_report.json").write_text(json.dumps(report,indent=2)+"\n")
    print(json.dumps(report,indent=2))








"""Private MARS 3D MRI foundation-model validation.

The candidate replaces 2D RadImageNet tokens with three volumetric embeddings
from the official Apache-2.0 MARS 3D Swin Transformer. All fitting uses only
non-gold report groups; the 58 organizer-labelled studies are evaluation-only.
"""

import subprocess
import sys
import urllib.request

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "monai==1.4.0"])
from monai.networks.nets.swin_unetr import SwinTransformer


MARS_URL = "https://drive.usercontent.google.com/download?id=1__lWJfBaCSQqkyPvxpQqK-MWH_d__bWz&export=download&confirm=t"
MARS_SHA256 = "88388a59d92935a15d30bd41c0e9dbece99ba78db73cb374d44290fd6566ccc3"
MARS_SIZE = 82121520
MARS_REPOSITORY = "zqiuak/MARS"
MARS_FEATURE_DIM = 768
VOLUME_DEPTH = 32
VOLUME_SIZE = 96


class MARSBackbone(nn.Module):
    def __init__(self):
        super().__init__()
        self.swinViT = SwinTransformer(
            in_chans=1,
            embed_dim=48,
            window_size=(7, 7, 7),
            patch_size=(2, 2, 2),
            depths=(2, 2, 2, 2),
            num_heads=(3, 6, 12, 24),
            mlp_ratio=4.0,
            qkv_bias=True,
            norm_layer=nn.LayerNorm,
            use_checkpoint=False,
            spatial_dims=3,
            use_v2=True,
        )

    def forward(self, volume):
        feature = self.swinViT(volume)[4]
        return torch.nn.functional.adaptive_avg_pool3d(feature, 1).flatten(1)


def load_mars():
    path = Path("/kaggle/working/MARS.pt")
    if not path.is_file():
        urllib.request.urlretrieve(MARS_URL, path)
    if path.stat().st_size != MARS_SIZE:
        raise RuntimeError("MARS checkpoint size drift")
    if sha256(path) != MARS_SHA256:
        raise RuntimeError("MARS checkpoint hash drift")
    state = torch.load(path, map_location="cpu", weights_only=True)
    model = MARSBackbone()
    incompatible = model.load_state_dict(state, strict=False)
    if incompatible.missing_keys or incompatible.unexpected_keys:
        raise RuntimeError(
            f"MARS state mismatch: missing={incompatible.missing_keys}, "
            f"unexpected={incompatible.unexpected_keys}"
        )
    model.eval().to(DEVICE)
    for parameter in model.parameters():
        parameter.requires_grad_(False)
    return model


def render_volume(record):
    if record is None:
        return np.zeros((VOLUME_DEPTH, VOLUME_SIZE, VOLUME_SIZE), np.float32), 0
    rows = ordered_headers(record)
    if not rows:
        return np.zeros((VOLUME_DEPTH, VOLUME_SIZE, VOLUME_SIZE), np.float32), 0
    source_indices = np.linspace(0, len(rows) - 1, VOLUME_DEPTH).round().astype(int)
    images = []
    for index in source_indices:
        try:
            image, spacing = read_pixels(rows[index][0])
            height, width = image.shape
            if spacing is None:
                spacing = np.asarray((0.5, 0.5))
            crop_h = min(height, max(16, int(round(140.0 / max(spacing[0], 1e-3)))))
            crop_w = min(width, max(16, int(round(140.0 / max(spacing[1], 1e-3)))))
            y0, x0 = (height - crop_h) // 2, (width - crop_w) // 2
            image = image[y0:y0 + crop_h, x0:x0 + crop_w]
            image = cv2.resize(image, (VOLUME_SIZE, VOLUME_SIZE), interpolation=cv2.INTER_AREA)
            images.append(image.astype(np.float32))
        except Exception:
            return np.zeros((VOLUME_DEPTH, VOLUME_SIZE, VOLUME_SIZE), np.float32), 0
    volume = np.stack(images)
    nonzero = volume[np.isfinite(volume) & (volume != 0)]
    if len(nonzero) < 100:
        return np.zeros_like(volume), 0
    mean, standard_deviation = float(nonzero.mean()), float(nonzero.std())
    if standard_deviation < 1e-6:
        return np.zeros_like(volume), 0
    foreground = np.isfinite(volume) & (volume != 0)
    normalized = np.zeros_like(volume, dtype=np.float32)
    normalized[foreground] = (volume[foreground] - mean) / standard_deviation
    normalized = np.clip(normalized, -8.0, 8.0).astype(np.float32)
    return normalized, 1


@torch.no_grad()
def encode_mars(studies, by_study):
    model = load_mars()
    features = np.zeros((len(studies), len(PLANES), MARS_FEATURE_DIM), np.float16)
    masks = np.zeros((len(studies), len(PLANES)), np.float32)
    batch_studies = 8
    for start in range(0, len(studies), batch_studies):
        ids = studies[start:start + batch_studies]
        jobs = []
        for uid in ids:
            records = by_study.get(uid, [])
            jobs.extend(choose(records, plane, True, True) for plane in PLANES)
        with ThreadPoolExecutor(max_workers=8) as pool:
            rendered = list(pool.map(render_volume, jobs))
        volumes = np.stack([item[0] for item in rendered])
        valid = np.asarray([item[1] for item in rendered], np.float32)
        encoded = np.zeros((len(volumes), MARS_FEATURE_DIM), np.float16)
        valid_indices = np.flatnonzero(valid > 0)
        for offset in range(0, len(valid_indices), 4):
            indices = valid_indices[offset:offset + 4]
            tensor = torch.from_numpy(volumes[indices]).to(DEVICE).unsqueeze(1)
            with torch.autocast("cuda", enabled=DEVICE.type == "cuda"):
                encoded[indices] = model(tensor).float().cpu().numpy().astype(np.float16)
        features[start:start + len(ids)] = encoded.reshape(len(ids), len(PLANES), MARS_FEATURE_DIM)
        masks[start:start + len(ids)] = valid.reshape(len(ids), len(PLANES))
        if start % (batch_studies * 20) == 0:
            log(f"MARS encoded {start}/{len(studies)}")
    del model
    gc.collect(); torch.cuda.empty_cache()
    return features, masks


class MARSAttentionHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.norm = nn.LayerNorm(MARS_FEATURE_DIM)
        self.proj = nn.Linear(MARS_FEATURE_DIM, 256)
        self.attention = nn.Linear(256, len(TARGETS))
        self.classifier = nn.Parameter(torch.empty(len(TARGETS), 256))
        self.bias = nn.Parameter(torch.zeros(len(TARGETS)))
        nn.init.trunc_normal_(self.classifier, std=0.02)

    def forward(self, features, mask):
        hidden = torch.nn.functional.gelu(self.proj(self.norm(features.float())))
        attention = self.attention(hidden)
        attention = attention.masked_fill(mask[:, :, None] < 0.5, -1e4)
        attention = torch.softmax(attention, dim=1)
        pooled = torch.einsum("btn,btf->bnf", attention, hidden)
        return (pooled * self.classifier).sum(-1) + self.bias


@torch.no_grad()
def mars_predict(model, features, masks, indices):
    model.eval(); output = []
    for start in range(0, len(indices), 128):
        index = indices[start:start + 128]
        x = torch.from_numpy(features[index]).to(DEVICE)
        mask = torch.from_numpy(masks[index]).to(DEVICE)
        output.append(torch.sigmoid(model(x, mask)).cpu().numpy())
    return np.concatenate(output)


def train_mars_head(features, masks, labels, train_index, validation_index, seed):
    seed_all(seed)
    model = MARSAttentionHead().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-3)
    dataset = TensorDataset(
        torch.from_numpy(features[train_index]),
        torch.from_numpy(masks[train_index]),
        torch.from_numpy(labels[train_index]),
    )
    loader = DataLoader(
        dataset, batch_size=64, shuffle=True,
        generator=torch.Generator().manual_seed(seed),
    )
    best_state, best_score = None, -1.0
    for _ in range(20):
        model.train()
        for x, mask, target in loader:
            x, mask, target = x.to(DEVICE), mask.to(DEVICE), target.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss = torch.nn.functional.binary_cross_entropy_with_logits(
                model(x, mask), target.float()
            )
            loss.backward(); optimizer.step()
        prediction = mars_predict(model, features, masks, validation_index)
        score = macro_auc(labels[validation_index], prediction)
        if score > best_score:
            best_score = score
            best_state = {key: value.detach().cpu().clone()
                          for key, value in model.state_dict().items()}
    model.load_state_dict(best_state)
    prediction = mars_predict(model, features, masks, np.arange(len(features)))
    del model
    torch.cuda.empty_cache()
    return prediction, float(best_score)


def normalized_reports_mars(series):
    return (series.fillna("").astype(str).str.lower()
            .str.replace(r"\s+", " ", regex=True).str.strip())


def report_groups_mars(series):
    return np.asarray([hashlib.sha256(text.encode()).hexdigest()[:24]
                       for text in normalized_reports_mars(series)])


def mars_3d_main():
    if DEVICE.type != "cuda":
        raise RuntimeError("MARS 3D validation requires GPU")
    root = competition_root()
    train = pd.read_csv(root / "train.csv", dtype={"StudyInstanceUID": str})
    studies = train.StudyInstanceUID.tolist()
    gold = train[TARGETS].notna().all(axis=1).to_numpy()
    if int(gold.sum()) != 58:
        raise RuntimeError(f"expected 58 gold studies, found {gold.sum()}")
    gold_index = np.flatnonzero(gold)
    non_gold = np.flatnonzero(~gold)

    weak = pd.read_csv(unique_input("llm_labels_v4_blend.csv"), dtype={"StudyInstanceUID": str})
    aligned = train[["StudyInstanceUID"]].merge(weak, on="StudyInstanceUID", how="left", validate="one_to_one")
    labels = aligned[TARGETS].to_numpy(np.float32)
    if not np.isfinite(labels).all():
        raise RuntimeError("weak labels incomplete")

    series = pd.read_csv(root / "train_series.csv", dtype=str)
    series_root = root / ("train_series" if (root / "train_series").is_dir() else "train_images")
    items = [(row.StudyInstanceUID, row.SeriesInstanceUID, row.Anatomical_Plane,
              series_root / row.StudyInstanceUID / row.SeriesInstanceUID)
             for row in series.itertuples(index=False)]
    with ThreadPoolExecutor(max_workers=16) as pool:
        records = [record for record in pool.map(inspect_series, items) if record]
    by_study = {}
    for record in records:
        by_study.setdefault(record["study"], []).append(record)
    mars_features, mars_masks = encode_mars(studies, by_study)
    if float(mars_masks.mean()) < 0.75:
        raise RuntimeError(f"MARS sequence coverage too low: {mars_masks.mean()}")
    missing_studies = int((mars_masks.sum(axis=1) == 0).sum())
    if missing_studies:
        raise RuntimeError(f"MARS has no valid sequence for {missing_studies} studies")

    package = np.load(unique_input("incumbent_radimagenet_features.npz"), allow_pickle=False)
    if package["ids"].astype(str).tolist() != studies:
        raise RuntimeError("incumbent feature identity/order drift")
    if str(package["encoder_sha256"]) != RAD_SHA:
        raise RuntimeError("incumbent encoder hash drift")
    control_features, control_masks = package["features"], package["masks"]

    groups = report_groups_mars(train["Report"])
    y_gold = train.loc[gold, TARGETS].to_numpy(float)
    results, control_gold, mars_gold = [], [], []
    for seed in SEEDS:
        unique_groups = np.unique(groups[non_gold])
        shuffled = np.random.default_rng(seed).permutation(unique_groups)
        validation_groups = set(shuffled[:max(1, int(0.10 * len(shuffled)))])
        validation_index = non_gold[np.asarray([group in validation_groups for group in groups[non_gold]])]
        train_index = non_gold[np.asarray([group not in validation_groups for group in groups[non_gold]])]
        if set(groups[train_index]).intersection(groups[validation_index]):
            raise RuntimeError("report-group leakage")
        control_prediction, control_weak = train_head(
            control_features, control_masks, labels, train_index, validation_index, seed
        )
        mars_prediction, mars_weak = train_mars_head(
            mars_features, mars_masks, labels, train_index, validation_index, seed
        )
        control_auc = macro_auc(y_gold, control_prediction[gold_index])
        mars_auc = macro_auc(y_gold, mars_prediction[gold_index])
        results.append({
            "seed": seed,
            "control_weak_validation_auc": control_weak,
            "mars_weak_validation_auc": mars_weak,
            "control_gold_auc": control_auc,
            "mars_gold_auc": mars_auc,
            "delta": mars_auc - control_auc,
        })
        control_gold.append(control_prediction[gold_index])
        mars_gold.append(mars_prediction[gold_index])
        log(f"seed {seed}: MARS 3D delta {mars_auc - control_auc:+.6f}")

    control_mean = np.mean(control_gold, axis=0)
    mars_mean = np.mean(mars_gold, axis=0)
    control_auc = macro_auc(y_gold, control_mean)
    mars_auc = macro_auc(y_gold, mars_mean)
    bootstrap = paired_bootstrap(y_gold, control_mean, mars_mean)
    deltas = np.asarray([result["delta"] for result in results])
    gate = {
        "delta_at_least_0_005": mars_auc - control_auc >= 0.005,
        "positive_seeds_at_least_4": int((deltas > 0).sum()) >= 4,
        "bootstrap_p_at_least_0_95": bootstrap["p_improve"] >= 0.95,
        "bootstrap_q025_positive": bootstrap["q025"] > 0,
    }
    report = {
        "contract": "private MARS 3D foundation validation; gold evaluation only; no submission",
        "candidate": "MARS 3D Swin Transformer Base",
        "candidate_repository": MARS_REPOSITORY,
        "candidate_license": "Apache-2.0",
        "candidate_citation": "Large-scale multi-sequence pretraining for generalizable MRI analysis, Nature Biomedical Engineering (2026)",
        "candidate_checkpoint_sha256": MARS_SHA256,
        "candidate_checkpoint_size": MARS_SIZE,
        "candidate_volume_shape": [VOLUME_DEPTH, VOLUME_SIZE, VOLUME_SIZE],
        "candidate_sequence_tokens": list(PLANES),
        "sequence_coverage": float(mars_masks.mean()),
        "control": "RadImageNet ResNet-50",
        "control_sha256": RAD_SHA,
        "weak_labels": "stevenleehans llm_labels_v4_blend.csv",
        "gold_fit_count": 0,
        "gold_evaluation_count": 58,
        "seeds": results,
        "ensemble": {
            "control_gold_auc": control_auc,
            "mars_gold_auc": mars_auc,
            "delta": mars_auc - control_auc,
        },
        "bootstrap": bootstrap,
        "gate": gate,
        "passed": bool(all(gate.values())),
        "submission_created": False,
    }
    Path("/kaggle/working/mars_3d_validation_report.json").write_text(
        json.dumps(report, indent=2) + "\n", encoding="utf-8"
    )
    print(json.dumps(report, indent=2))


mars_3d_main()

